# 01 - Prepare training/validation data (UrbanSARFloods)

**Settings:** Accelerator = None (CPU), Internet = On. Run with *Save Version > Save & Run All*.

Streams the 38 GB archive from Hugging Face (never stored), converts every chip to compact uint8
GeoTIFFs, adds HAND / slope / permanent-water layers and computes normalisation statistics.
Output: `/kaggle/working/ufm-trainval` (~17 GB). Attach this notebook's output as input to notebook 03.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"   # <- edit
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import os, subprocess
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q rasterio 2>/dev/null; python -c "import rasterio; print('rasterio', rasterio.__version__)"

In [ ]:
!python scripts/prepare_train.py --out /kaggle/working/ufm-trainval

In [ ]:
!python scripts/build_aux.py --root /kaggle/working/ufm-trainval --cache /tmp/tiles --workers 4 --max-cache-gb 6

In [ ]:
!python scripts/compute_stats.py --root /kaggle/working/ufm-trainval

In [ ]:
!rm -f /kaggle/working/ufm-trainval/_stable_samples.npz
!du -sh /kaggle/working/ufm-trainval
!for d in /kaggle/working/ufm-trainval/*/*/; do echo "$d $(ls $d | wc -l)"; done